# Record identifiers and external cohort flow

Run after `data_preparation.ipynb` and `evaluation.ipynb` (which writes `results_evaluation/patient_split.npz`).

1. Prints the external-cohort screening counts for Figure 1B.
2. Writes `appendix16_record_ids_split.csv` (ZZU pECG record and patient identifiers with the split assignment: fold1-fold5 or test) for Multimedia Appendix 16.

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_DIR = "rebuild_v3"
SAVE_DIR = "results_evaluation"

for name in ("korea_pairing_flow.csv", "korea_flow.csv"):
    print(f"\n{name}")
    print(pd.read_csv(os.path.join(DATA_DIR, name)).to_string(index=False))

In [ ]:
dev = np.load(os.path.join(DATA_DIR, "china_v2.npz"), allow_pickle=True)
split = np.load(os.path.join(SAVE_DIR, "patient_split.npz"), allow_pickle=True)

ids = dev["ids"].astype(str)
split_label = np.full(len(ids), "", dtype=object)
split_label[np.isin(ids, split["test_pts"].astype(str))] = "test"
for k in range(1, 6):
    split_label[np.isin(ids, split[f"val_pts_fold{k}"].astype(str))] = f"fold{k}"
assert (split_label != "").all()

records = pd.DataFrame({
    "record_id": dev["ecg_ids"].astype(str),
    "patient_id": ids,
    "split": split_label,
})
records.to_csv("appendix16_record_ids_split.csv", index=False)
print(records["split"].value_counts().sort_index())
print(records.groupby("split")["patient_id"].nunique())